# 06 — The targeting policy

**Tenday**

Everything upstream produced inputs. This produces the **decision**.

Two outputs, kept deliberately separate:

| Output | Uses | Answers |
|---|---|---|
| **Retrospective verdict** | Realised outcomes | Which channels and offers paid back? |
| **Forward tiers** | Day-30 predictions only | What should we do with each new customer? |

Mixing them is the easy mistake: scoring a customer on realised value isn't a
prediction, it's a description.

## Install

In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 MB 138.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Dependencies from notebook 05

The out-of-fold step needs `encode()` and the Layer 2 feature list. Copied here so
this notebook runs on its own.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

L1_NUMERIC = [
    "credit_limit", "bonus_amount", "minimum_spend", "acquisition_cost",
    "offer_richness", "threshold_to_limit_ratio",
]
L1_CATEGORICAL = [
    "acquisition_channel", "campaign_id", "campaign_type",
    "offer_type", "age_band", "income_band", "region",
]

# First-30-day behaviour. Every column prefixed d30_ by construction --
# the prefix IS the horizon guarantee.
L2_BEHAVIOUR = [
    "d30_total_spend", "d30_avg_txn", "d30_max_txn", "d30_txn_count",
    "d30_category_hhi", "d30_liquid_share", "d30_online_share",
    "d30_recurring_share", "d30_active_day_ratio", "d30_dormancy_days",
    "d30_distinct_categories", "d30_distinct_merchants",
    "d30_spend_velocity", "d30_spend_per_active_day", "d30_refund_count",
    "d30_qualified", "d30_days_to_qualify", "d30_redeemed",
    "d30_redemption_lag", "d30_bonus_earned",
]

NEVER_HAPPENED = {
    "d30_days_to_qualify": 30.0,     # did not qualify within the window
    "d30_redemption_lag": 30.0,
}


def load(spark, out, write_format="delta"):
    l2 = spark.read.format(write_format).load(f"{out}/features_l2")
    l3 = spark.read.format(write_format).load(f"{out}/features_l3")
    lab = spark.read.format(write_format).load(f"{out}/hidden_labels")

    target = l3.select("customer_id", "net_value", "is_profitable")
    df = (l2.join(target, "customer_id")
            .join(lab.select("customer_id", "is_gamer"), "customer_id"))
    return df.toPandas()


def encode(pdf):
    """One-hot the categoricals; fill 'did not happen' semantics."""
    df = pdf.copy()
    for c, v in NEVER_HAPPENED.items():
        if c in df.columns:
            df[c] = df[c].fillna(v)
    for c in L2_BEHAVIOUR:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    dummies = pd.get_dummies(df[L1_CATEGORICAL], prefix=L1_CATEGORICAL,
                             drop_first=True)
    df = pd.concat([df.drop(columns=L1_CATEGORICAL), dummies], axis=1)
    l1_cols = L1_NUMERIC + list(dummies.columns)
    return df, l1_cols

## Setup and tier definitions

In [ ]:
"""
Tenday -- Notebook 06: the targeting policy.

Everything upstream produces inputs; this produces the decision.

Inputs
------
  segments_named    behavioural segment per customer   (Layer 3, KMeans)
  gaming_scores     anomaly score and flag             (Layer 3, IForest)
  features_l3       realised net value and economics
  features_l2       day-30 features -> predicted value (Layer 2, XGBoost)

Two distinct outputs, and the distinction matters
-------------------------------------------------
  1. RETROSPECTIVE VERDICT -- for each channel x campaign combination,
     what did it actually produce? This is what steers next quarter's
     acquisition budget. It uses realised outcomes, which is legitimate
     because the spend already happened.

  2. FORWARD TIERS -- for customers currently in their first 30 days,
     rank on predicted value against gaming risk. This is the operational
     output, and it uses only day-30 information because that is all a
     live customer has produced.

Conflating the two is the easiest mistake here: using realised value to
"score" a customer is not a prediction, it is a description.
"""

import numpy as np
import pandas as pd
from pyspark.sql import functions as F


TIERS = [
    # (tier, label, action)
    (1, "Priority 1 -- Acquire",          "High predicted value, low gaming risk"),
    (2, "Priority 2 -- Acquire & monitor", "High predicted value, elevated risk"),
    (3, "Priority 3 -- Acquire on lean offers", "Moderate value, low risk"),
    (4, "Watch list",                      "Ambiguous -- early-tenure monitoring"),
    (5, "Do not target",                   "Low value, high gaming risk"),
]

## Step 1 — assemble the scored book

Joins segment, gaming score, realised value and prediction into one frame per customer.

In [ ]:
def assemble(spark, out, gaming_segment, predicted_value=None,
             write_format="delta"):
    """
    Join every model output into one customer-level frame.

    `predicted_value` is an optional pandas frame of
    (customer_id, predicted_value) from the Layer 2 model. When absent,
    realised net_value is used and the output is retrospective only.
    """
    l1  = spark.read.format(write_format).load(f"{out}/features_l1")
    l3  = spark.read.format(write_format).load(f"{out}/features_l3")
    seg = spark.read.format(write_format).load(f"{out}/segments_named")
    gs  = spark.read.format(write_format).load(f"{out}/gaming_scores")

    df = (l1.select("customer_id", "acquisition_channel", "campaign_id",
                    "campaign_type", "offer_type", "acquisition_cost",
                    "bonus_amount", "income_band", "region")
            .join(l3.select("customer_id", "net_value", "gross_revenue",
                            "is_profitable", "full_bonus_earned",
                            "full_total_spend", "full_days_to_qualify"),
                  "customer_id")
            .join(seg, "customer_id")
            .join(gs, "customer_id"))

    df = df.withColumn("is_flagged_gaming",
                       (F.col("segment") == F.lit(gaming_segment)).cast("int"))

    if predicted_value is not None:
        pv = spark.createDataFrame(predicted_value)
        df = df.join(pv, "customer_id", "left")

    return df

## Step 2 — the retrospective verdict

For every channel × campaign pair: how many customers, what share gamed, what it cost, what came back.

In [ ]:
def channel_offer_verdict(book, min_customers=500):
    """
    The number that moves budget. For every channel x campaign pair:
    how many customers, what share gamed, what did it cost, what came back.
    """
    v = (book.groupBy("acquisition_channel", "campaign_id", "campaign_type")
             .agg(F.count("*").alias("customers"),
                  F.avg("is_flagged_gaming").alias("gaming_rate"),
                  F.sum("acquisition_cost").alias("total_acq_cost"),
                  F.sum("full_bonus_earned").alias("total_bonus_paid"),
                  F.sum("gross_revenue").alias("total_revenue"),
                  F.sum("net_value").alias("total_net_value"),
                  F.avg("net_value").alias("avg_net_value"))
             .filter(F.col("customers") >= min_customers))

    v = (v.withColumn("cost_per_acquisition",
                      (F.col("total_acq_cost") + F.col("total_bonus_paid"))
                      / F.col("customers"))
          .withColumn("roi",
                      F.col("total_revenue")
                      / (F.col("total_acq_cost") + F.col("total_bonus_paid")))
          .orderBy(F.desc("avg_net_value")))
    return v


def gaming_cost_summary(book):
    """Headline economics: what gaming cost, and who carries the book."""
    return (book.groupBy("segment_name")
                .agg(F.count("*").alias("customers"),
                     F.round(F.avg("net_value"), 0).alias("avg_net_value"),
                     F.round(F.sum("net_value") / 1e7, 2).alias("total_net_value_cr"),
                     F.round(F.sum("full_bonus_earned") / 1e7, 2).alias("bonus_paid_cr"),
                     F.round(F.sum("acquisition_cost") / 1e7, 2).alias("acq_cost_cr"))
                .orderBy(F.desc("total_net_value_cr")))

## Step 3 — forward tiers

Two axes — predicted value and gaming risk — into five actions. Thresholds are
**percentiles**, not fixed rupee amounts, so the policy travels across books of
different size. They're a business lever, not a fitted parameter.

In [ ]:
def assign_tiers(book, value_col="predicted_value",
                 value_hi=0.66, value_lo=0.33, risk_hi=0.66):
    """
    Two axes -- predicted value and gaming risk -- into five tiers.

    Thresholds are percentile-based rather than absolute so the policy
    travels across books of different size and mix. They are a business
    lever, not a fitted parameter: a bank with more review capacity
    lowers risk_hi and inspects more accounts.
    """
    qs = book.approxQuantile(value_col, [value_lo, value_hi], 0.01)
    v_lo, v_hi = qs[0], qs[1]
    r_hi = book.approxQuantile("gaming_score", [risk_hi], 0.01)[0]

    high_v = F.col(value_col) >= F.lit(v_hi)
    mid_v = (F.col(value_col) >= F.lit(v_lo)) & (F.col(value_col) < F.lit(v_hi))
    high_r = (F.col("gaming_score") >= F.lit(r_hi)) | (F.col("is_flagged_gaming") == 1)

    tier = (F.when(high_v & ~high_r, F.lit(1))
             .when(high_v & high_r, F.lit(2))
             .when(mid_v & ~high_r, F.lit(3))
             .when(mid_v & high_r, F.lit(4))
             .otherwise(F.lit(5)))

    labels = F.lit("Do not target")
    for t, label, _ in TIERS:
        labels = F.when(F.col("tier") == t, F.lit(label)).otherwise(labels)

    out = (book.withColumn("tier", tier)
               .withColumn("tier_label", labels))
    print(f"value thresholds: p{int(value_lo*100)}={v_lo:,.0f}  "
          f"p{int(value_hi*100)}={v_hi:,.0f}   risk p{int(risk_hi*100)}={r_hi:.3f}")
    return out


def tier_summary(tiered):
    return (tiered.groupBy("tier", "tier_label")
                  .agg(F.count("*").alias("customers"),
                       F.round(F.avg("net_value"), 0).alias("actual_avg_net_value"),
                       F.round(F.avg("is_flagged_gaming"), 3).alias("gaming_rate"),
                       F.round(F.sum("net_value") / 1e7, 2).alias("total_net_value_cr"))
                  .orderBy("tier"))

## Step 4 — policy simulation

**A bound, not a forecast.** Customers declined under a live policy wouldn't be
identical to these, and changed offer terms would change behaviour.

In [ ]:
def simulate_policy(book, exclude_tiers=(5,), tier_col="tier"):
    """
    Counterfactual: if the tiers had been applied and the excluded groups
    never acquired, what changes?

    Stated as a bound, not a promise. The customers you decline are not
    identical to the ones you would have declined had the policy been
    live, and offer terms would shift behaviour. This measures the size
    of the prize, not a guaranteed saving.
    """
    tot = book.agg(F.sum("net_value").alias("net"),
                   F.count("*").alias("n"),
                   F.sum("acquisition_cost").alias("acq"),
                   F.sum("full_bonus_earned").alias("bonus")).collect()[0]

    kept = (book.filter(~F.col(tier_col).isin(list(exclude_tiers)))
                .agg(F.sum("net_value").alias("net"),
                     F.count("*").alias("n"),
                     F.sum("acquisition_cost").alias("acq"),
                     F.sum("full_bonus_earned").alias("bonus")).collect()[0])

    return pd.DataFrame([
        {"scenario": "Actual book",
         "customers": tot["n"],
         "net_value_cr": round(tot["net"] / 1e7, 2),
         "acq_spend_cr": round((tot["acq"] + tot["bonus"]) / 1e7, 2)},
        {"scenario": f"Excluding tier {list(exclude_tiers)}",
         "customers": kept["n"],
         "net_value_cr": round(kept["net"] / 1e7, 2),
         "acq_spend_cr": round((kept["acq"] + kept["bonus"]) / 1e7, 2)},
    ])

---

# Run

## Out-of-fold predictions

Tiering needs a prediction for **every** customer — but a model always looks
better on data it was trained on. So each customer is scored by a model that
never saw them: split into five folds, predict each fold from the other four.

In [ ]:
OUT = "/Volumes/workspace/default/tenday"
pdf = load(spark, OUT, "delta")
print(pdf.shape)

In [0]:
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
import xgboost as xgb

OUT = "/Volumes/workspace/default/tenday"

df_enc, l1_cols = encode(pdf)
l2_cols = l1_cols + [c for c in L2_BEHAVIOUR if c in df_enc.columns]

X = df_enc[l2_cols].values
y = df_enc["net_value"].values
oof = np.zeros(len(df_enc))

for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(X):
    m = xgb.XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1).fit(X[tr_i], y[tr_i])
    oof[te_i] = m.predict(X[te_i])

pv = pd.DataFrame({"customer_id": df_enc["customer_id"].astype("int64"),
                   "predicted_value": oof.astype("float64")})
print("out-of-fold R²:", round(np.corrcoef(oof, y)[0, 1] ** 2, 3))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5701134581657464>, line 7
      3 import xgboost as xgb
      5 OUT = "/Volumes/workspace/default/tenday"
----> 7 df_enc, l1_cols = encode(pdf)
      8 l2_cols = l1_cols + [c for c in L2_BEHAVIOUR if c in df_enc.columns]
     10 X = df_enc[l2_cols].values

NameError: name 'encode' is not defined

Out-of-fold R² of **0.605**, against 0.658 on the held-out split. Small gap — the model generalises.

## Apply the policy

`gaming_segment=4` must match *Bonus Opportunists* from notebook 03. KMeans IDs
are arbitrary — re-check if clustering was re-run.

In [0]:
book = assemble(spark, OUT, gaming_segment=4, predicted_value=pv, write_format="delta")

print("=== segment economics ===")
gaming_cost_summary(book).show(truncate=False)

print("=== channel x campaign, best ===")
v = channel_offer_verdict(book, min_customers=500)
v.select("acquisition_channel", "campaign_id", "customers", "gaming_rate",
         "avg_net_value", "cost_per_acquisition", "roi").show(8, False)

print("=== channel x campaign, worst ===")
v.orderBy("avg_net_value").select("acquisition_channel", "campaign_id",
         "gaming_rate", "avg_net_value", "roi").show(8, False)

tiered = assign_tiers(book)
tier_summary(tiered).show(truncate=False)
print(simulate_policy(tiered).to_string(index=False))

tiered.select("customer_id", "segment_name", "gaming_score", "predicted_value",
              "net_value", "tier", "tier_label") \
      .write.format("delta").mode("overwrite").save(f"{OUT}/targeting")

## Targeting policy — results

### Segment economics (200,000 customers, ₹51.2cr total bonus paid)

| Segment | Customers | Avg net value | Total (₹cr) | Bonus paid (₹cr) |
|---|---|---|---|---|
| High-Value Core | 84,517 | +₹3,154 | **+26.65** | 30.97 |
| Borderline Opportunists | 5,040 | −₹1,091 | −0.55 | 0.14 |
| Minimal Engagers | 9,734 | −₹1,133 | −1.10 | 0.00 |
| Early Churners | 27,718 | −₹708 | −1.96 | 1.07 |
| Casual Regulars | 45,120 | −₹709 | −3.20 | 1.31 |
| **Bonus Opportunists** | 27,871 | **−₹2,676** | **−7.46** | **17.74** |

Bonus Opportunists are 13.9% of customers and absorbed **34.6% of all
bonus spend**, returning −₹7.46cr.

### The primary finding: offer structure, not channel

| | Best channel | Worst channel | Spread |
|---|---|---|---|
| C007 (₹2,000 bonus / ₹45k threshold) | +₹3,030 | +₹2,799 | ₹231 |
| C003 (₹12,000 bonus / ₹150k threshold) | −₹1,562 | −₹2,121 | ₹559 |

Within-campaign variation across channels is ₹231–₹559. Between-campaign
variation is roughly **₹5,000 per customer**.

C007 is profitable through every channel including affiliate. C003 and
C006 lose money through every channel including branch and referral. ROI
is ~2.9 for lean offers and 0.69–0.75 for rich ones.

Rich offers do not attract better customers. They attract people who will
do precisely enough to qualify and then stop. Gaming rate reaches 26.7%
on affiliate/C003 against 2.5% on branch/C007 — but the campaign axis
explains far more of that than the channel axis.

**Implication: restructure the offer before reallocating the budget.**

### Forward tiers

| Tier | Action | Customers | Actual avg net value | Gaming rate |
|---|---|---|---|---|
| 1 | Acquire | 36,178 | +₹2,117 | 0.000 |
| 2 | Acquire & monitor | 32,905 | **+₹7,936** | 0.199 |
| 3 | Acquire on lean offers | 49,116 | −₹335 | 0.000 |
| 4 | Watch list | 15,741 | −₹161 | 0.244 |
| 5 | Do not target | 66,060 | −₹2,950 | 0.264 |

Tier 2 carries the highest realised value **and** an elevated gaming
rate — high value and high risk overlap because both involve heavy
spend. A pure risk screen would reject the most profitable cohort in the
book. This is why the tier action is monitoring rather than exclusion.

### Policy simulation — a bound, not a forecast

| Scenario | Customers | Net value (₹cr) | Acquisition spend (₹cr) |
|---|---|---|---|
| Actual book | 200,000 | 12.39 | 78.06 |
| Excluding tier 5 | 133,940 | 31.87 | 38.46 |

**Read this carefully.** Tier 5 is 33% of the book and only 26.4% of it
is gamers, so most of the improvement comes from withholding bonuses to
low-value customers generally — not from gaming prevention alone.
Declining a third of applicants is also not a realistic policy.

The simulation sizes the prize; it does not promise it. The customers
declined under a live policy would not be identical to these, and
changed offer terms would change behaviour.

---

## Next

**`07_automated_insights`** — have an LLM write the stakeholder memo, then check
every number it uses against the computed results.